# Target Tail Exploration

This notebook focuses on one question:

- how heavy-tailed is `next_sa_ipv4_count_delta` for one router, and what do those tails look like visually?

The plots are intentionally simple and dependency-light so the notebook can run directly in this repo.

In [ ]:
from __future__ import annotations

import html
import math
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, SVG, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_feature_frame
from src.targets import add_next_window_targets


def _svg_text(x: float, y: float, text: str, size: int = 12, anchor: str = 'start', weight: str = 'normal') -> str:
    return (
        f"<text x='{x:.1f}' y='{y:.1f}' font-size='{size}' text-anchor='{anchor}' "
        f"font-family='IBM Plex Sans, Arial, sans-serif' font-weight='{weight}'>{html.escape(text)}</text>"
    )


def make_hist_svg(
    series: pd.Series,
    title: str,
    subtitle: str,
    bins: int = 48,
    clip_quantile: float | None = None,
    width: int = 900,
    height: int = 360,
) -> str:
    values = series.dropna().astype(float)
    if values.empty:
        return "<svg xmlns='http://www.w3.org/2000/svg' width='900' height='80'></svg>"

    if clip_quantile is not None:
        low = values.quantile(1 - clip_quantile)
        high = values.quantile(clip_quantile)
        clipped = values.clip(lower=low, upper=high)
        note = f'clipped to [{low:.0f}, {high:.0f}]'
    else:
        clipped = values
        low = clipped.min()
        high = clipped.max()
        note = 'full range'

    if math.isclose(low, high):
        low -= 1.0
        high += 1.0

    counts, edges = pd.cut(clipped, bins=bins, retbins=True, include_lowest=True)
    bin_counts = counts.value_counts(sort=False).tolist()
    max_count = max(bin_counts) if bin_counts else 1

    margin_left = 70
    margin_right = 24
    margin_top = 58
    margin_bottom = 54
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    bar_width = plot_width / bins

    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<rect width='100%' height='100%' fill='#fffdf8'/>",
        _svg_text(24, 28, title, size=20, weight='600'),
        _svg_text(24, 47, f"{subtitle} | {note}", size=12),
        f"<line x1='{margin_left}' y1='{margin_top + plot_height}' x2='{margin_left + plot_width}' y2='{margin_top + plot_height}' stroke='#444' stroke-width='1'/>",
        f"<line x1='{margin_left}' y1='{margin_top}' x2='{margin_left}' y2='{margin_top + plot_height}' stroke='#444' stroke-width='1'/>",
    ]

    for tick_ratio in (0.0, 0.25, 0.5, 0.75, 1.0):
        y = margin_top + plot_height * (1.0 - tick_ratio)
        tick_value = int(round(max_count * tick_ratio))
        parts.append(f"<line x1='{margin_left}' y1='{y:.1f}' x2='{margin_left + plot_width}' y2='{y:.1f}' stroke='#ece7dc' stroke-width='1'/>")
        parts.append(_svg_text(margin_left - 10, y + 4, str(tick_value), size=11, anchor='end'))

    for index, count in enumerate(bin_counts):
        x = margin_left + index * bar_width
        bar_height = 0.0 if max_count == 0 else plot_height * count / max_count
        y = margin_top + plot_height - bar_height
        parts.append(
            f"<rect x='{x + 0.5:.1f}' y='{y:.1f}' width='{max(bar_width - 1, 1):.1f}' height='{max(bar_height, 1):.1f}' fill='#19647e' opacity='0.88'/>"
        )

    for value, label in ((0.0, '0'), (low, f'{low:.0f}'), (high, f'{high:.0f}')):
        if value < low or value > high:
            continue
        ratio = (value - low) / (high - low)
        x = margin_left + ratio * plot_width
        parts.append(f"<line x1='{x:.1f}' y1='{margin_top}' x2='{x:.1f}' y2='{margin_top + plot_height}' stroke='#a6a6a6' stroke-dasharray='4 4' stroke-width='1'/>")
        parts.append(_svg_text(x, height - 18, label, size=11, anchor='middle'))

    parts.append(_svg_text(width / 2, height - 4, 'target value', size=12, anchor='middle'))
    parts.append(_svg_text(18, margin_top + plot_height / 2, 'count', size=12))
    parts.append('</svg>')
    return ''.join(parts)


def make_ccdf_svg(
    series: pd.Series,
    title: str,
    subtitle: str,
    width: int = 900,
    height: int = 360,
) -> str:
    values = series.dropna().astype(float).abs()
    values = values[values > 0].sort_values().reset_index(drop=True)
    if values.empty:
        return "<svg xmlns='http://www.w3.org/2000/svg' width='900' height='80'></svg>"

    tail_prob = pd.Series(1.0 - (values.index + 1) / len(values))
    tail_prob = tail_prob.clip(lower=1 / len(values))
    log_x = values.map(math.log10)
    log_y = tail_prob.map(math.log10)

    x_min, x_max = log_x.min(), log_x.max()
    y_min, y_max = log_y.min(), 0.0
    if math.isclose(x_min, x_max):
        x_min -= 1.0
        x_max += 1.0

    margin_left = 70
    margin_right = 24
    margin_top = 58
    margin_bottom = 54
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom

    def scale_x(value: float) -> float:
        return margin_left + (value - x_min) / (x_max - x_min) * plot_width

    def scale_y(value: float) -> float:
        return margin_top + (y_max - value) / (y_max - y_min) * plot_height

    points = ' '.join(
        f"{scale_x(x):.1f},{scale_y(y):.1f}"
        for x, y in zip(log_x, log_y, strict=True)
    )

    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<rect width='100%' height='100%' fill='#fffdf8'/>",
        _svg_text(24, 28, title, size=20, weight='600'),
        _svg_text(24, 47, subtitle, size=12),
        f"<line x1='{margin_left}' y1='{margin_top + plot_height}' x2='{margin_left + plot_width}' y2='{margin_top + plot_height}' stroke='#444' stroke-width='1'/>",
        f"<line x1='{margin_left}' y1='{margin_top}' x2='{margin_left}' y2='{margin_top + plot_height}' stroke='#444' stroke-width='1'/>",
    ]

    for power in range(math.floor(x_min), math.ceil(x_max) + 1):
        x = scale_x(power)
        label = f"1e{power}" if power != 0 else '1'
        parts.append(f"<line x1='{x:.1f}' y1='{margin_top}' x2='{x:.1f}' y2='{margin_top + plot_height}' stroke='#ece7dc' stroke-width='1'/>")
        parts.append(_svg_text(x, height - 18, label, size=11, anchor='middle'))

    for power in range(math.floor(y_min), 1):
        y = scale_y(power)
        label = f"1e{power}" if power != 0 else '1'
        parts.append(f"<line x1='{margin_left}' y1='{y:.1f}' x2='{margin_left + plot_width}' y2='{y:.1f}' stroke='#ece7dc' stroke-width='1'/>")
        parts.append(_svg_text(margin_left - 10, y + 4, label, size=11, anchor='end'))

    parts.append(f"<polyline fill='none' stroke='#c84c09' stroke-width='2' points='{points}'/>")
    parts.append(_svg_text(width / 2, height - 4, '|delta|', size=12, anchor='middle'))
    parts.append(_svg_text(18, margin_top + plot_height / 2, 'tail prob', size=12))
    parts.append('</svg>')
    return ''.join(parts)


def make_threshold_bar_svg(frame: pd.DataFrame, title: str, width: int = 900, height: int = 320) -> str:
    margin_left = 90
    margin_right = 24
    margin_top = 58
    margin_bottom = 54
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    max_value = frame['share'].max()
    row_width = plot_width / max(len(frame), 1)

    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<rect width='100%' height='100%' fill='#fffdf8'/>",
        _svg_text(24, 28, title, size=20, weight='600'),
        _svg_text(24, 47, 'share of rows with absolute delta above each threshold', size=12),
        f"<line x1='{margin_left}' y1='{margin_top + plot_height}' x2='{margin_left + plot_width}' y2='{margin_top + plot_height}' stroke='#444' stroke-width='1'/>",
        f"<line x1='{margin_left}' y1='{margin_top}' x2='{margin_left}' y2='{margin_top + plot_height}' stroke='#444' stroke-width='1'/>",
    ]

    for tick_ratio in (0.0, 0.25, 0.5, 0.75, 1.0):
        y = margin_top + plot_height * (1.0 - tick_ratio)
        tick_value = max_value * tick_ratio
        parts.append(f"<line x1='{margin_left}' y1='{y:.1f}' x2='{margin_left + plot_width}' y2='{y:.1f}' stroke='#ece7dc' stroke-width='1'/>")
        parts.append(_svg_text(margin_left - 10, y + 4, f'{tick_value:.0%}', size=11, anchor='end'))

    for index, row in frame.reset_index(drop=True).iterrows():
        x = margin_left + index * row_width
        bar_height = 0.0 if max_value == 0 else plot_height * row['share'] / max_value
        y = margin_top + plot_height - bar_height
        parts.append(
            f"<rect x='{x + 8:.1f}' y='{y:.1f}' width='{max(row_width - 16, 10):.1f}' height='{max(bar_height, 1):.1f}' fill='#3d9970' opacity='0.9'/>"
        )
        parts.append(_svg_text(x + row_width / 2, height - 18, f"{int(row['threshold'])}", size=11, anchor='middle'))
        parts.append(_svg_text(x + row_width / 2, y - 8, f"{row['share']:.1%}", size=11, anchor='middle'))

    parts.append(_svg_text(width / 2, height - 4, '|delta| threshold', size=12, anchor='middle'))
    parts.append('</svg>')
    return ''.join(parts)


## Load One Router / One Target

This keeps the notebook aligned with the modeling runs we have been discussing.

In [ ]:
DATABASE_PATH = REPO_ROOT / 'data/2025-03-01-to-2026-03-31/netflow_window.sqlite'
ROUTER = 'oh_ir1_gw'
TARGET = 'next_sa_ipv4_count_delta'
BASE_COLUMN = 'sa_ipv4_count'

frame = build_feature_frame(DATABASE_PATH, train_router=ROUTER)
frame = add_next_window_targets(frame)
frame = frame.dropna(subset=[TARGET]).reset_index(drop=True)
frame['timestamp_dt'] = pd.to_datetime(frame['timestamp'], unit='s', utc=True)

target = frame[TARGET].astype(float)
abs_target = target.abs()

display(Markdown(f"**Rows:** {len(frame):,}  \\n**Router:** `{ROUTER}`  \\n**Target:** `{TARGET}`  \\n**Database:** `{DATABASE_PATH.name}`"))

In [ ]:
quantiles = target.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
summary = pd.DataFrame(
    {
        'metric': [
            'rows',
            'mean',
            'std',
            'min',
            '1%',
            '5%',
            '25%',
            '50%',
            '75%',
            '95%',
            '99%',
            'max',
            'mean |delta|',
            'share |delta| > 1000',
            'share |delta| > 5000',
            'share delta > 0',
        ],
        'value': [
            len(target),
            target.mean(),
            target.std(),
            target.min(),
            quantiles.loc[0.01],
            quantiles.loc[0.05],
            quantiles.loc[0.25],
            quantiles.loc[0.50],
            quantiles.loc[0.75],
            quantiles.loc[0.95],
            quantiles.loc[0.99],
            target.max(),
            abs_target.mean(),
            (abs_target > 1000).mean(),
            (abs_target > 5000).mean(),
            (target > 0).mean(),
        ],
    }
)
summary

## Distribution Views

Two useful views:

- a clipped histogram to show where most rows live
- a tail plot of `|delta|` to show how quickly extreme events fall off

The second one is often more informative for this project because the extreme deltas dominate training difficulty.

In [ ]:
display(SVG(make_hist_svg(
    target,
    title='Target Histogram',
    subtitle='Most mass is near zero, but the tails are still very wide',
    clip_quantile=0.995,
)))

display(SVG(make_ccdf_svg(
    target,
    title='Absolute-Delta Tail Curve',
    subtitle='Log-log CCDF of |delta|. A slow falloff means heavy tails.',
)))

In [ ]:
threshold_frame = pd.DataFrame(
    {
        'threshold': [100, 250, 500, 1000, 2500, 5000, 10000],
    }
)
threshold_frame['share'] = threshold_frame['threshold'].map(lambda value: float((abs_target > value).mean()))

display(SVG(make_threshold_bar_svg(
    threshold_frame,
    title='How Often Large Deltas Happen',
)))

threshold_frame

## Largest Tail Events

This table helps connect the abstract tail metrics back to concrete windows in the dataset.

In [ ]:
largest_events = frame[
    ['timestamp_dt', BASE_COLUMN, f'next_{BASE_COLUMN}', TARGET]
].copy()
largest_events['abs_delta'] = largest_events[TARGET].abs()
largest_events = largest_events.sort_values('abs_delta', ascending=False).head(20)
largest_events = largest_events.rename(
    columns={
        BASE_COLUMN: 'current_count',
        f'next_{BASE_COLUMN}': 'next_count',
        TARGET: 'delta',
        'timestamp_dt': 'timestamp_utc',
    }
)
largest_events.reset_index(drop=True)

## Interpretation Notes

- If the histogram is tightly centered but the tail curve decays slowly, the target is innovation-heavy: most windows are modest, but rare events dominate error.
- If `|delta| > 1000` still happens often, a model can improve MAE while still struggling badly on `R²`, because a few large misses contribute a lot of squared error.
- The largest-events table is a reminder that this is not a smooth forecasting target. The biggest jumps are the exact rows that decide whether a model looks merely decent or genuinely useful.